## 01. 추천 시스템

<aside>
💡

**추천 시스템**은 사용자에게 적합한 아이템(영화, 제품, 음악 등)을 추천하는 알고리즘이다. 오늘날 다양한 분야에서 사용되며, 사용자와 아이템 간의 **선호도**나 **유사성**을 기반으로 맞춤형 추천을 제공한다.

</aside>

- 추천 시스템의 주요 유형
    1. **콘텐츠 기반 필터링(Content-Based Filtering)**: 아이템의 속성을 기반으로 사용자에게 적합한 아이템을 추천
    2. **협업 필터링(Collaborative Filtering)**: 사용자들 간의 유사성을 기반으로 추천
    3. **하이브리드 추천 시스템(Hybrid Recommendation Systems)**: 협업 필터링과 콘텐츠 기반 필터링을 결합하여 추천

## 02. 콘텐츠 기반 필터링 (Content-Based Filtering)

### 02-01. 콘텐츠 기반 필터링 개요

<aside>
💡

**콘텐츠 기반 필터링**은 아이템의 속성을 분석하여 사용자가 선호할 만한 아이템을 추천하는 방식이다. 

예) 사용자가 좋아한 영화의 **장르**, **감독**, **배우** 등의 특성을 바탕으로 유사한 속성을 가진 아이템을 추천한다.

</aside>

In [2]:
import pandas as pd
import numpy as np

In [7]:
# 간단한 영화 데이터 생성
movie_data = {
    'movie_id': [101, 102, 103, 104, 105],
    'title': ['Movie A', 'Movie B', 'Movie C', 'Movie D', 'Movie E'],
    'genre': ['Action', 'Comedy', 'Action', 'Comedy', 'Drama']
}

user_ratings = {
    'user_id': [1, 1, 1, 2, 2],
    'movie_id': [101, 102, 103, 104, 105],
    'rating': [5, 3, 4, 4, 5]
}

movie_df = pd.DataFrame(movie_data)
user_ratings_df = pd.DataFrame(user_ratings)

In [9]:
# 장르를 one-hot encoding하여 콘텐츠 기반 행렬 생성
movie_df['genre_encoded'] = pd.get_dummies(movie_df['genre']).values.tolist()

# 특정 사용자가 평가한 영화의 장르 벡터를 가져옴
target_user_ratings = user_ratings_df[user_ratings_df['user_id'] == 1]

# 해당 사용자가 평가한 영화의 상세 정보 추출
rated_movies = movie_df[movie_df['movie_id'].isin(target_user_ratings['movie_id'])]

# [1,0,0] 또는 [0,0,1] 이러한 값의 벡터 평균 계산 = 어떤 장르를 주로 봤는지 (장르벡터)
user_genre_profile = np.mean(
    np.array(rated_movies['genre_encoded'].tolist()),
    axis=0)

# 나머지 영화와의 유사도 계산 (dot 값이 클수록 사용자의 선호장르와 유사한 컨텐츠)
movie_df['similarity'] = movie_df['genre_encoded'].apply(
    lambda x: np.dot(user_genre_profile, x))   # dot로 벡터 내적을 구해(코사인 유사도)

movie_df

,movie_id,title,genre,genre_encoded,similarity
0,101,Movie A,Action,"[True, False, False]",0.666667
1,102,Movie B,Comedy,"[False, True, False]",0.333333
2,103,Movie C,Action,"[True, False, False]",0.666667
3,104,Movie D,Comedy,"[False, True, False]",0.333333
4,105,Movie E,Drama,"[False, False, True]",0.000000


In [10]:

# 추천할 영화 선택 (사용자가 보지 않은 영화)
recommendations = movie_df[~movie_df['movie_id'].isin(target_user_ratings['movie_id'])].sort_values(by='similarity', ascending=False)

print(f"Recommendations for user 1 based on content: {recommendations[['title', 'similarity']]}")

Recommendations for user 1 based on content:      title  similarity
3  Movie D    0.333333
4  Movie E    0.000000


콘텐츠 기반 추천 시스템
- 내가 봤던, 혹은 좋아하는 아이템의 특징을 바탕으로 비슷한 아이템을 추천한다.
- 여기서는 각 영화의 장르 벡터와 사용자 취향 프로필의 유사도를 계산하고
- 사용자가 아직 보지 않은 영화중 유사도가 높은 영화를 추천한다.

콘첸츠 기반 추천 시스템 = 아이템 중신
- 내가 좋아하는 아이템의 속성과 비슷한 속성을 가진 아이템을 추천

협업 필터링 추천시스템 = 사용자 중심
- 나랑 비슷한 사람이 좋아하는 아이템을 추천
- 데이터가 없으면 실용성이 없을 수 있다.

## 03. 협업 필터링 (Collaborative Filtering)

### 03-01. 협업 필터링 개요

<aside>
💡

**협업 필터링(Collaborative Filtering)**은 사용자 간의 상호작용 데이터를 이용하여 아이템을 추천하는 방식이다.

</aside>

- **사용자 기반(User-based)**: 비슷한 취향을 가진 다른 사용자들이 선호하는 아이템을 추천
- **아이템 기반(Item-based)**: 비슷한 속성을 가진 아이템 간의 유사성을 바탕으로 추천

03-02. 협업 필터링 예제 (사용자 기반 협업 필터링)

In [12]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 간단한 사용자-아이템 평점 데이터 생성
data = {
    'user_id': [1, 1, 1, 2, 2, 3, 3, 4, 4, 4],
    'item_id': [101, 102, 103, 101, 104, 101, 102, 103, 104, 105],
    'rating': [5, 3, 4, 4, 5, 3, 4, 2, 4, 5]
}

df = pd.DataFrame(data)

df

,user_id,item_id,rating
0,1,101,5
1,1,102,3
2,1,103,4
3,2,101,4
4,2,104,5
5,3,101,3
6,3,102,4
7,4,103,2
8,4,104,4
9,4,105,5


In [13]:
user_item_matrix = df.pivot_table(index='user_id', columns='item_id', values='rating').fillna(0)

user_item_matrix

item_id,101,102,103,104,105
user_id,,,,,
1,5.0,3.0,4.0,0.0,0.0
2,4.0,0.0,0.0,5.0,0.0
3,3.0,4.0,0.0,0.0,0.0
4,0.0,0.0,2.0,4.0,5.0


In [ ]:
# 사용자 간 평점 패턴으로 유사도 계산 (코사인 유사도)
user_similarity = cosine_similarity(user_item_matrix)

user_similarity

array([[1.        , 0.4417261 , 0.76367532, 0.16865481],
       [0.4417261 , 1.        , 0.37481703, 0.4656202 ],
       [0.76367532, 0.37481703, 1.        , 0.        ],
       [0.16865481, 0.4656202 , 0.        , 1.        ]])

In [ ]:
# 행과 열을 둘다 user_id로 지정하여 어떤 사용자끼리 유사한지를 확인

In [15]:
# 행과 열을 둘다 user_id로 지정하여 어떤 사용자끼리 유사한지를 확인
user_similarity_df = pd.DataFrame(user_similarity, index=user_item_matrix.index, columns=user_item_matrix.index)

user_similarity_df

user_id,1,2,3,4
user_id,,,,
1,1.000000,0.441726,0.763675,0.168655
2,0.441726,1.000000,0.374817,0.465620
3,0.763675,0.374817,1.000000,0.000000
4,0.168655,0.465620,0.000000,1.000000


In [ ]:
# 특정 사용자의 추천 아이템 (예: user_id=1)
target_user = 1    # 자기 자신은 유사도 1로 가장 높으므로 제외

# targer_user와 가장 유사한 유저를 추출
similar_users = user_similarity_df[target_user].sort_values(ascending=False).index[1:]

similar_users

Index([3, 2, 4], dtype='int64', name='user_id')

In [17]:
user_item_matrix

item_id,101,102,103,104,105
user_id,,,,,
1,5.0,3.0,4.0,0.0,0.0
2,4.0,0.0,0.0,5.0,0.0
3,3.0,4.0,0.0,0.0,0.0
4,0.0,0.0,2.0,4.0,5.0


In [18]:

# 유사한 사용자의 아이템 중에서, target_user가 평가하지 않은 아이템 추천

# 해당 사용자가 평가한 아이템 목록
items_rated_by_target = user_item_matrix.loc[
    target_user,
    user_item_matrix.loc[target_user] > 0   # 열을 해당 유저의 전체 평점을 가져와 평점이 0보다 큰 아이템 선택
    ].index         # item_id

items_rated_by_target

Index([101, 102, 103], dtype='int64', name='item_id')

In [19]:
recommendations = []    # 추천 아이템 저장 리스트

for user in similar_users:
    items = user_item_matrix.loc[user, user_item_matrix.loc[user] > 0].index    # 비슷한 유저가 평가한 아이템
    
    # target_user 가 아직 평가하지 않은 아이템만 추천 아이템으로 선정 
    new_recommendations = [item for item in items if item not in items_rated_by_target]
    
    recommendations.extend(new_recommendations)     # 추천 후보를 추천 리스트에 추가
    
    if len(recommendations) >= 2:   # 최대 2개까지 추천받도록
        break

print(f"Recommendations for user {target_user}: {recommendations}")

Recommendations for user 1: [104, 104, 105]


협업 필터링
- 이번예제에서는 사용자의 평점 패턴을 비교하고, 비슷한 사용자가 좋아한 아이템을 추천하는 방식을 만들었다.
- 사용자 - 아이템 평점 행렬을 만든 후, 코사인 유사도로 비슷한 사용자를 찾는다.
- 이후 target_user 가 아직 평가(관람, 구매) 하지 않은 아이템을 추천해준다